# Concurrency, Standard Library & Performance — Senior Developer Interview Prep

Master the standard library power tools, concurrency models, type hints, testing patterns, and performance optimization.

**Topics Covered:**
1. Collections Module
2. Itertools
3. Functools
4. Threading & the GIL
5. Multiprocessing
6. Asyncio
7. Type Hints & Generics
8. Testing Patterns
9. Performance Optimization
10. Practice Problems

In [ ]:
import sys
print(f"Python version: {sys.version}")

---
## 1. Collections Module

The `collections` module provides specialized container types that are often more efficient and readable than plain dicts/lists.

In [ ]:
from collections import defaultdict, Counter, deque, OrderedDict, ChainMap

# defaultdict — auto-initializes missing keys
# Real-world: building an adjacency list for a graph
edges = [("A", "B"), ("A", "C"), ("B", "D"), ("C", "D"), ("D", "E")]

graph = defaultdict(list)
for src, dst in edges:
    graph[src].append(dst)
    graph[dst].append(src)  # undirected

print("Adjacency list:")
for node, neighbors in sorted(graph.items()):
    print(f"  {node} → {neighbors}")

In [ ]:
# defaultdict with int/set
word_count = defaultdict(int)
for word in "the cat sat on the mat the cat".split():
    word_count[word] += 1
print(f"Word counts: {dict(word_count)}")

# Group by first letter
by_letter = defaultdict(set)
for word in ["apple", "banana", "avocado", "blueberry", "cherry", "apricot"]:
    by_letter[word[0]].add(word)
print(f"By letter: {dict(by_letter)}")

In [ ]:
# Counter — count occurrences, find most common
# Real-world: analyzing log levels, error frequencies

log_levels = ["ERROR", "INFO", "ERROR", "WARN", "INFO", "INFO", "ERROR", "DEBUG", "INFO", "ERROR"]

counts = Counter(log_levels)
print(f"Counts: {counts}")
print(f"Most common: {counts.most_common(2)}")
print(f"ERROR count: {counts['ERROR']}")

# Counter arithmetic
morning = Counter(INFO=10, ERROR=3, WARN=1)
afternoon = Counter(INFO=15, ERROR=7, WARN=2)
total = morning + afternoon
print(f"\nTotal: {total}")
print(f"Difference: {afternoon - morning}")  # only positive counts

In [ ]:
# deque — O(1) append/pop from both ends
# Real-world: sliding window, BFS, bounded history

# Bounded history buffer
history = deque(maxlen=5)
for i in range(10):
    history.append(f"event_{i}")
print(f"Last 5 events: {list(history)}")

# Sliding window maximum
from collections import deque

def sliding_window_max(nums, k):
    """Find max in each window of size k using a monotonic deque."""
    dq = deque()  # stores indices
    result = []
    for i, num in enumerate(nums):
        while dq and dq[0] <= i - k:
            dq.popleft()
        while dq and nums[dq[-1]] <= num:
            dq.pop()
        dq.append(i)
        if i >= k - 1:
            result.append(nums[dq[0]])
    return result

print(f"\nSliding max (k=3): {sliding_window_max([1,3,-1,-3,5,3,6,7], 3)}")

In [ ]:
# ChainMap — search through multiple dicts in order
# Real-world: layered configuration (env vars → config file → defaults)

defaults = {"theme": "dark", "lang": "en", "debug": False, "log_level": "INFO"}
config_file = {"theme": "light", "log_level": "DEBUG"}
env_vars = {"debug": True}

config = ChainMap(env_vars, config_file, defaults)  # first dict has highest priority

print(f"theme:     {config['theme']}")      # from config_file
print(f"debug:     {config['debug']}")      # from env_vars
print(f"lang:      {config['lang']}")       # from defaults
print(f"log_level: {config['log_level']}")  # from config_file

---
## 2. Itertools

The `itertools` module provides fast, memory-efficient building blocks for iteration. These come up frequently in interviews.

In [ ]:
import itertools

# chain — flatten multiple iterables
merged = list(itertools.chain([1, 2], [3, 4], [5, 6]))
print(f"chain: {merged}")

# chain.from_iterable — flatten a list of lists
nested = [[1, 2], [3, 4], [5, 6]]
flat = list(itertools.chain.from_iterable(nested))
print(f"flatten: {flat}")

In [ ]:
# product — cartesian product (nested loops in one line)
# Real-world: generating test combinations

sizes = ["S", "M", "L"]
colors = ["Red", "Blue"]
materials = ["Cotton", "Polyester"]

variants = list(itertools.product(sizes, colors, materials))
print(f"Total variants: {len(variants)}")
for v in variants[:4]:
    print(f"  {v}")
print("  ...")

In [ ]:
# combinations, permutations
from itertools import combinations, permutations

# Real-world: finding all possible pairs from a team
team = ["Alice", "Bob", "Charlie", "Diana"]

pairs = list(combinations(team, 2))  # order doesn't matter
print(f"Possible pairs ({len(pairs)}):")
for p in pairs:
    print(f"  {p[0]} & {p[1]}")

# All possible orderings of 3 items
print(f"\nPermutations of [1,2,3]: {list(permutations([1,2,3]))}")

In [ ]:
# groupby — group consecutive elements
from itertools import groupby

# IMPORTANT: data must be sorted by the key first!
transactions = [
    {"type": "credit", "amount": 100},
    {"type": "credit", "amount": 200},
    {"type": "debit", "amount": 50},
    {"type": "debit", "amount": 75},
    {"type": "credit", "amount": 150},
]

sorted_txns = sorted(transactions, key=lambda t: t["type"])
for txn_type, group in groupby(sorted_txns, key=lambda t: t["type"]):
    items = list(group)
    total = sum(t["amount"] for t in items)
    print(f"{txn_type}: {len(items)} transactions, total=${total}")

In [ ]:
# islice — slice any iterator (no need to convert to list)
from itertools import islice, count, accumulate

# Take first 5 from an infinite counter
print(f"First 5 from count(10): {list(islice(count(10), 5))}")

# accumulate — running totals
prices = [10, 20, 30, 40, 50]
running_total = list(accumulate(prices))
print(f"Prices:        {prices}")
print(f"Running total: {running_total}")

# Running max
import operator
data = [3, 1, 4, 1, 5, 9, 2, 6]
running_max = list(accumulate(data, max))
print(f"Data:        {data}")
print(f"Running max: {running_max}")

In [ ]:
# Real-world: pairwise iteration (sliding window of 2)
from itertools import pairwise  # Python 3.10+

prices = [100, 102, 98, 105, 103, 110]
changes = [(b - a) for a, b in pairwise(prices)]
print(f"Prices:  {prices}")
print(f"Changes: {changes}")

# For Python < 3.10, use this recipe:
def pairwise_compat(iterable):
    a, b = itertools.tee(iterable)
    next(b, None)
    return zip(a, b)

In [ ]:
# zip_longest — zip with fill value for unequal lengths
from itertools import zip_longest

names = ["Alice", "Bob", "Charlie"]
scores = [95, 87]

print("zip (truncates):")
print(list(zip(names, scores)))

print("\nzip_longest (fills):")
print(list(zip_longest(names, scores, fillvalue="N/A")))

---
## 3. Functools

Higher-order functions and operations on callable objects.

In [ ]:
import functools

# lru_cache — automatic memoization with LRU eviction
@functools.lru_cache(maxsize=256)
def expensive_computation(n):
    """Simulates an expensive function."""
    import time
    time.sleep(0.01)  # simulate work
    return n ** 2

import time

start = time.perf_counter()
results = [expensive_computation(i) for i in range(50)]
first_run = time.perf_counter() - start

start = time.perf_counter()
results = [expensive_computation(i) for i in range(50)]
cached_run = time.perf_counter() - start

print(f"First run:  {first_run:.3f}s")
print(f"Cached run: {cached_run:.6f}s")
print(f"Cache info: {expensive_computation.cache_info()}")

In [ ]:
# partial — freeze some function arguments
# Real-world: creating specialized versions of generic functions

from functools import partial
import json

pretty_json = partial(json.dumps, indent=2, sort_keys=True)

data = {"name": "Alice", "age": 30, "city": "NYC"}
print(pretty_json(data))

# Partial with logging
import logging
log_error = partial(print, "[ERROR]")  # simplified for demo
log_info = partial(print, "[INFO]")

log_error("Connection failed")
log_info("Server started")

In [ ]:
# reduce — fold a sequence to a single value
from functools import reduce

# Flatten nested list
nested = [[1, 2], [3, 4], [5, 6]]
flat = reduce(lambda acc, x: acc + x, nested)
print(f"Flattened: {flat}")

# Build a dict from pairs
pairs = [("a", 1), ("b", 2), ("c", 3)]
result = reduce(lambda d, kv: {**d, kv[0]: kv[1]}, pairs, {})
print(f"Dict: {result}")

# Compose functions
def compose(*funcs):
    return reduce(lambda f, g: lambda x: f(g(x)), funcs)

transform = compose(str.upper, str.strip, str.title)
print(f"Composed: '{transform('  hello world  ')}'")

In [ ]:
# singledispatch — function overloading by type
from functools import singledispatch

@singledispatch
def process(data):
    raise TypeError(f"Unsupported type: {type(data)}")

@process.register(str)
def _(data):
    return f"String: {data.upper()}"

@process.register(list)
def _(data):
    return f"List with {len(data)} items: {data}"

@process.register(dict)
def _(data):
    return f"Dict with keys: {list(data.keys())}"

@process.register(int)
@process.register(float)
def _(data):
    return f"Number: {data * 2}"

print(process("hello"))
print(process([1, 2, 3]))
print(process({"a": 1}))
print(process(42))
print(process(3.14))

In [ ]:
# total_ordering — fill in missing comparison methods
from functools import total_ordering

@total_ordering
class Version:
    def __init__(self, version_string):
        self.parts = tuple(int(x) for x in version_string.split("."))
    
    def __eq__(self, other):
        return self.parts == other.parts
    
    def __lt__(self, other):
        return self.parts < other.parts
    
    def __repr__(self):
        return f"Version('{'.'.join(str(p) for p in self.parts)}')" 

versions = [Version("2.1.0"), Version("1.9.5"), Version("2.0.1"), Version("1.10.0")]
print(f"Sorted: {sorted(versions)}")
print(f"Max: {max(versions)}")
print(f"2.1.0 >= 2.0.1? {Version('2.1.0') >= Version('2.0.1')}")

---
## 4. Threading & the GIL

Python's Global Interpreter Lock (GIL) means only one thread executes Python bytecode at a time. Threading is useful for **I/O-bound** tasks, not CPU-bound.

In [ ]:
import threading
import time

# I/O-bound task — threading helps here
def fetch_url(url, results, index):
    """Simulate an HTTP request."""
    time.sleep(0.5)  # simulates network I/O
    results[index] = f"Response from {url}"

urls = [f"https://api.example.com/data/{i}" for i in range(5)]

# Sequential
start = time.perf_counter()
results_seq = [None] * len(urls)
for i, url in enumerate(urls):
    fetch_url(url, results_seq, i)
seq_time = time.perf_counter() - start

# Threaded
start = time.perf_counter()
results_thr = [None] * len(urls)
threads = []
for i, url in enumerate(urls):
    t = threading.Thread(target=fetch_url, args=(url, results_thr, i))
    threads.append(t)
    t.start()
for t in threads:
    t.join()
thr_time = time.perf_counter() - start

print(f"Sequential: {seq_time:.2f}s")
print(f"Threaded:   {thr_time:.2f}s")
print(f"Speedup:    {seq_time/thr_time:.1f}x")

In [ ]:
# concurrent.futures — higher-level API (preferred in production)
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

def process_item(item):
    time.sleep(0.2)  # simulate work
    return item ** 2

items = list(range(10))

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as executor:
    # Method 1: map — preserves order
    results = list(executor.map(process_item, items))
elapsed = time.perf_counter() - start

print(f"Results: {results}")
print(f"Time: {elapsed:.2f}s (vs {len(items) * 0.2:.1f}s sequential)")

In [ ]:
# Thread safety — race conditions and locks
import threading

class BankAccount:
    def __init__(self, balance=0):
        self.balance = balance
        self._lock = threading.Lock()
    
    def transfer_unsafe(self, amount):
        """NOT thread-safe — read-modify-write race condition."""
        current = self.balance
        time.sleep(0.001)  # simulate processing
        self.balance = current + amount
    
    def transfer_safe(self, amount):
        """Thread-safe with lock."""
        with self._lock:
            current = self.balance
            time.sleep(0.001)
            self.balance = current + amount

# Demonstrate race condition
def run_transfers(account, method, n=100):
    threads = []
    for _ in range(n):
        t = threading.Thread(target=method, args=(1,))
        threads.append(t)
        t.start()
    for t in threads:
        t.join()
    return account.balance

unsafe_account = BankAccount(0)
result = run_transfers(unsafe_account, unsafe_account.transfer_unsafe, 50)
print(f"Unsafe (expected 50): {result}")

safe_account = BankAccount(0)
result = run_transfers(safe_account, safe_account.transfer_safe, 50)
print(f"Safe (expected 50):   {result}")

---
## 5. Multiprocessing

For **CPU-bound** tasks, use multiprocessing to bypass the GIL. Each process gets its own Python interpreter.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import time
import os

def cpu_bound_work(n):
    """CPU-intensive: compute sum of squares."""
    return sum(i * i for i in range(n))

data = [5_000_000] * 8

# Sequential
start = time.perf_counter()
results_seq = [cpu_bound_work(n) for n in data]
seq_time = time.perf_counter() - start
print(f"Sequential: {seq_time:.2f}s")

# Parallel with ProcessPoolExecutor
start = time.perf_counter()
with ProcessPoolExecutor(max_workers=4) as executor:
    results_par = list(executor.map(cpu_bound_work, data))
par_time = time.perf_counter() - start
print(f"Parallel:   {par_time:.2f}s")
print(f"Speedup:    {seq_time/par_time:.1f}x")
print(f"CPU cores:  {os.cpu_count()}")

In [ ]:
# When to use what:
# 
# | Task Type  | Best Tool               | Why                                    |
# |------------|-------------------------|----------------------------------------|
# | I/O-bound  | threading / asyncio     | GIL released during I/O waits          |
# | CPU-bound  | multiprocessing         | Separate processes, each with own GIL  |
# | Mixed      | ProcessPool + ThreadPool| Combine both as needed                 |
# | Simple I/O | asyncio                 | Single-threaded, event loop, efficient |

print("Threading vs Multiprocessing vs Asyncio:")
print("  Threading:       Best for I/O-bound (network, file I/O)")
print("  Multiprocessing: Best for CPU-bound (computation)")
print("  Asyncio:         Best for high-concurrency I/O (web servers, API clients)")

---
## 6. Asyncio

Single-threaded concurrency using coroutines. Perfect for high-concurrency I/O operations like web servers and API clients.

In [ ]:
import asyncio

# Basic coroutine
async def fetch_data(url, delay):
    """Simulate an async HTTP request."""
    print(f"  Fetching {url}...")
    await asyncio.sleep(delay)  # non-blocking sleep
    return {"url": url, "data": f"response from {url}"}

async def main():
    # Sequential (slow)
    start = asyncio.get_event_loop().time()
    r1 = await fetch_data("api/users", 0.5)
    r2 = await fetch_data("api/orders", 0.5)
    r3 = await fetch_data("api/products", 0.5)
    seq_time = asyncio.get_event_loop().time() - start
    print(f"  Sequential: {seq_time:.2f}s\n")

    # Concurrent (fast) — gather runs coroutines concurrently
    start = asyncio.get_event_loop().time()
    r1, r2, r3 = await asyncio.gather(
        fetch_data("api/users", 0.5),
        fetch_data("api/orders", 0.5),
        fetch_data("api/products", 0.5),
    )
    conc_time = asyncio.get_event_loop().time() - start
    print(f"  Concurrent: {conc_time:.2f}s")
    print(f"  Speedup: {seq_time/conc_time:.1f}x")

# In Jupyter, use await directly (event loop is already running)
await main()

In [ ]:
# asyncio.TaskGroup (Python 3.11+) — structured concurrency

async def process(name, duration):
    await asyncio.sleep(duration)
    if name == "failing":
        raise ValueError(f"Task {name} failed!")
    return f"{name} done"

async def main_taskgroup():
    results = []
    async with asyncio.TaskGroup() as tg:
        t1 = tg.create_task(process("fast", 0.1))
        t2 = tg.create_task(process("medium", 0.3))
        t3 = tg.create_task(process("slow", 0.5))
    
    print(f"All done: {t1.result()}, {t2.result()}, {t3.result()}")

await main_taskgroup()

In [ ]:
# Async generators & async context managers

async def async_range(start, stop, delay=0.1):
    """An async generator that yields values with delays."""
    for i in range(start, stop):
        await asyncio.sleep(delay)
        yield i

async def main_async_gen():
    values = []
    async for val in async_range(0, 5, 0.05):
        values.append(val)
    print(f"Async generator values: {values}")
    
    # Async comprehension
    squared = [val ** 2 async for val in async_range(0, 5, 0.05)]
    print(f"Async comprehension: {squared}")

await main_async_gen()

In [ ]:
# Real-world pattern: async semaphore for rate limiting

async def rate_limited_fetch(sem, url):
    async with sem:  # limits concurrent requests
        print(f"  Fetching {url}")
        await asyncio.sleep(0.3)  # simulate request
        return f"data from {url}"

async def main_semaphore():
    sem = asyncio.Semaphore(3)  # max 3 concurrent requests
    urls = [f"page/{i}" for i in range(9)]
    
    start = asyncio.get_event_loop().time()
    results = await asyncio.gather(
        *(rate_limited_fetch(sem, url) for url in urls)
    )
    elapsed = asyncio.get_event_loop().time() - start
    
    print(f"\n  Completed {len(results)} requests in {elapsed:.2f}s")
    print(f"  (9 requests, 3 at a time, 0.3s each ≈ 0.9s)")

await main_semaphore()

---
## 7. Type Hints & Generics

Type hints improve code readability and enable static analysis. Senior developers should be comfortable with advanced type annotations.

In [ ]:
from typing import (
    Optional, Union, TypeVar, Generic, Callable,
    TypeAlias, Literal, TypeGuard
)

# Basic type hints
def greet(name: str, excited: bool = False) -> str:
    greeting = f"Hello, {name}"
    return f"{greeting}!!!" if excited else greeting

# Optional — shorthand for Union[X, None]
def find_user(user_id: int) -> Optional[dict]:
    users = {1: {"name": "Alice"}, 2: {"name": "Bob"}}
    return users.get(user_id)

# Union and modern syntax (Python 3.10+)
def process(data: int | str | list) -> str:
    return str(data)

print(greet("World", excited=True))
print(find_user(1))
print(find_user(99))

In [ ]:
# TypeVar & Generic — create type-safe reusable containers
from typing import TypeVar, Generic

T = TypeVar('T')

class Stack(Generic[T]):
    """A type-safe stack."""
    def __init__(self) -> None:
        self._items: list[T] = []
    
    def push(self, item: T) -> None:
        self._items.append(item)
    
    def pop(self) -> T:
        if not self._items:
            raise IndexError("Stack is empty")
        return self._items.pop()
    
    def peek(self) -> T:
        if not self._items:
            raise IndexError("Stack is empty")
        return self._items[-1]
    
    def __len__(self) -> int:
        return len(self._items)
    
    def __repr__(self) -> str:
        return f"Stack({self._items})"

int_stack: Stack[int] = Stack()
int_stack.push(1)
int_stack.push(2)
int_stack.push(3)
print(f"Stack: {int_stack}")
print(f"Pop: {int_stack.pop()}")
print(f"Peek: {int_stack.peek()}")

In [ ]:
# Callable type hints
from typing import Callable

def apply_operation(
    data: list[int],
    operation: Callable[[int], int],  # takes int, returns int
    predicate: Callable[[int], bool] = lambda x: True
) -> list[int]:
    return [operation(x) for x in data if predicate(x)]

numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
result = apply_operation(numbers, lambda x: x ** 2, lambda x: x % 2 == 0)
print(f"Squares of evens: {result}")

In [ ]:
# Literal — restrict to specific values
from typing import Literal

def set_log_level(level: Literal["DEBUG", "INFO", "WARNING", "ERROR"]) -> None:
    print(f"Log level set to: {level}")

set_log_level("DEBUG")  # OK
# set_log_level("TRACE")  # type checker would flag this

# TypeAlias — name complex types
JSON: TypeAlias = dict[str, "JSON"] | list["JSON"] | str | int | float | bool | None

def parse_config(raw: str) -> JSON:
    import json
    return json.loads(raw)

result = parse_config('{"key": [1, 2, {"nested": true}]}')
print(f"Parsed: {result}")

---
## 8. Testing Patterns

Senior developers should write tests as naturally as production code. Here are key patterns.

In [ ]:
# Unit testing with unittest (built-in)
import unittest

class Calculator:
    def add(self, a, b):
        return a + b
    
    def divide(self, a, b):
        if b == 0:
            raise ValueError("Cannot divide by zero")
        return a / b

class TestCalculator(unittest.TestCase):
    def setUp(self):
        self.calc = Calculator()
    
    def test_add_positive(self):
        self.assertEqual(self.calc.add(2, 3), 5)
    
    def test_add_negative(self):
        self.assertEqual(self.calc.add(-1, -1), -2)
    
    def test_divide(self):
        self.assertAlmostEqual(self.calc.divide(10, 3), 3.333, places=3)
    
    def test_divide_by_zero(self):
        with self.assertRaises(ValueError) as ctx:
            self.calc.divide(10, 0)
        self.assertIn("zero", str(ctx.exception))

# Run in notebook
suite = unittest.TestLoader().loadTestsFromTestCase(TestCalculator)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)
print()  # clean output

In [ ]:
# Mocking — isolate units from dependencies
from unittest.mock import Mock, patch, MagicMock

class UserService:
    def __init__(self, db, email_client):
        self.db = db
        self.email_client = email_client
    
    def create_user(self, name, email):
        user = self.db.insert({"name": name, "email": email})
        self.email_client.send_welcome(email)
        return user

# Create mocks for dependencies
mock_db = Mock()
mock_db.insert.return_value = {"id": 1, "name": "Alice", "email": "alice@test.com"}

mock_email = Mock()

service = UserService(mock_db, mock_email)
result = service.create_user("Alice", "alice@test.com")

# Assert the right things were called
mock_db.insert.assert_called_once_with({"name": "Alice", "email": "alice@test.com"})
mock_email.send_welcome.assert_called_once_with("alice@test.com")
print(f"Created user: {result}")
print("All mock assertions passed!")

In [ ]:
# patch — temporarily replace objects during tests
from unittest.mock import patch
import os

def get_config():
    return {
        "debug": os.environ.get("DEBUG", "false") == "true",
        "db_url": os.environ.get("DATABASE_URL", "sqlite:///default.db"),
    }

# patch.dict replaces env vars for the test
with patch.dict(os.environ, {"DEBUG": "true", "DATABASE_URL": "postgres://prod"}):
    config = get_config()
    assert config["debug"] is True
    assert config["db_url"] == "postgres://prod"
    print(f"Inside patch: {config}")

config = get_config()
print(f"After patch:  {config}")

In [ ]:
# Parametrized tests pattern (without pytest)
import unittest

def is_palindrome(s: str) -> bool:
    cleaned = ''.join(c.lower() for c in s if c.isalnum())
    return cleaned == cleaned[::-1]

class TestPalindrome(unittest.TestCase):
    test_cases = [
        ("racecar", True),
        ("hello", False),
        ("A man, a plan, a canal: Panama", True),
        ("", True),
        ("ab", False),
        ("Was it a car or a cat I saw?", True),
    ]

def make_test(input_str, expected):
    def test_method(self):
        self.assertEqual(is_palindrome(input_str), expected,
                        f"is_palindrome('{input_str}') should be {expected}")
    return test_method

for i, (input_str, expected) in enumerate(TestPalindrome.test_cases):
    test_name = f"test_palindrome_{i}_{input_str[:20]}"
    setattr(TestPalindrome, test_name, make_test(input_str, expected))

suite = unittest.TestLoader().loadTestsFromTestCase(TestPalindrome)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)
print()

---
## 9. Performance Optimization

Know how to profile, measure, and optimize Python code.

In [ ]:
# __slots__ — reduce memory per instance, faster attribute access
import sys

class PointRegular:
    def __init__(self, x, y, z):
        self.x = x
        self.y = y
        self.z = z

class PointSlots:
    __slots__ = ('x', 'y', 'z')
    def __init__(self, x, y, z):
        self.x = x
        self.y = y
        self.z = z

regular = PointRegular(1, 2, 3)
slotted = PointSlots(1, 2, 3)

print(f"Regular: {sys.getsizeof(regular) + sys.getsizeof(regular.__dict__)} bytes")
print(f"Slots:   {sys.getsizeof(slotted)} bytes")
print(f"Regular has __dict__: {hasattr(regular, '__dict__')}")
print(f"Slots has __dict__:   {hasattr(slotted, '__dict__')}")

In [ ]:
# String concatenation performance
import time

n = 100_000

# BAD: string concatenation in a loop — O(n²)
start = time.perf_counter()
result = ""
for i in range(n):
    result += str(i)
concat_time = time.perf_counter() - start

# GOOD: join — O(n)
start = time.perf_counter()
result = "".join(str(i) for i in range(n))
join_time = time.perf_counter() - start

print(f"Concatenation: {concat_time:.3f}s")
print(f"Join:          {join_time:.3f}s")
print(f"Join is {concat_time/join_time:.1f}x faster")

In [ ]:
# Profiling with cProfile
import cProfile
import io
import pstats

def fibonacci_slow(n):
    if n < 2:
        return n
    return fibonacci_slow(n-1) + fibonacci_slow(n-2)

# Profile the function
profiler = cProfile.Profile()
profiler.enable()
fibonacci_slow(25)
profiler.disable()

stream = io.StringIO()
stats = pstats.Stats(profiler, stream=stream).sort_stats('cumulative')
stats.print_stats(5)
print(stream.getvalue())

In [ ]:
# Key performance tips for interviews

import time

n = 1_000_000

# 1. Use built-in functions — they're implemented in C
data = list(range(n))

start = time.perf_counter()
total_loop = 0
for x in data:
    total_loop += x
loop_time = time.perf_counter() - start

start = time.perf_counter()
total_builtin = sum(data)
builtin_time = time.perf_counter() - start

print(f"for loop: {loop_time:.4f}s")
print(f"sum():    {builtin_time:.4f}s")
print(f"Built-in is {loop_time/builtin_time:.1f}x faster")

In [ ]:
# 2. Local variable access is faster than global/attribute access
import math
import time

data = list(range(1, 500_000))

# Slow: attribute lookup each iteration
start = time.perf_counter()
result = [math.sqrt(x) for x in data]
attr_time = time.perf_counter() - start

# Faster: localize the function
start = time.perf_counter()
sqrt = math.sqrt  # bind to local
result = [sqrt(x) for x in data]
local_time = time.perf_counter() - start

print(f"math.sqrt: {attr_time:.4f}s")
print(f"local sqrt: {local_time:.4f}s")
print(f"Local is {attr_time/local_time:.1f}x faster")

In [ ]:
# 3. Avoid repeated dict/attribute lookups
import time

class Config:
    threshold = 0.5

data = [0.1 * i for i in range(1_000_000)]

# Slow: repeated attribute access
start = time.perf_counter()
result = [x for x in data if x > Config.threshold]
slow_time = time.perf_counter() - start

# Fast: cache the attribute
start = time.perf_counter()
threshold = Config.threshold
result = [x for x in data if x > threshold]
fast_time = time.perf_counter() - start

print(f"Repeated lookup: {slow_time:.4f}s")
print(f"Cached lookup:   {fast_time:.4f}s")

---
## 10. Practice Problems

### Problem 1: Implement an LRU Cache from scratch

Build an `LRUCache(capacity)` class with O(1) `get` and `put` operations.

```python
cache = LRUCache(2)
cache.put(1, "a")
cache.put(2, "b")
cache.get(1)       # returns "a"
cache.put(3, "c")  # evicts key 2 (least recently used)
cache.get(2)       # returns -1 (not found)
```

In [ ]:
# YOUR SOLUTION HERE


In [ ]:
# SOLUTION — using OrderedDict for O(1) operations
from collections import OrderedDict

class LRUCache:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.cache = OrderedDict()

    def get(self, key):
        if key not in self.cache:
            return -1
        self.cache.move_to_end(key)  # mark as recently used
        return self.cache[key]

    def put(self, key, value):
        if key in self.cache:
            self.cache.move_to_end(key)
        self.cache[key] = value
        if len(self.cache) > self.capacity:
            self.cache.popitem(last=False)  # remove oldest

    def __repr__(self):
        return f"LRUCache({dict(self.cache)})"

cache = LRUCache(2)
cache.put(1, "a")
cache.put(2, "b")
print(f"get(1): {cache.get(1)}")
cache.put(3, "c")
print(f"get(2): {cache.get(2)}")  # evicted
print(f"get(3): {cache.get(3)}")
print(f"Cache: {cache}")

### Problem 2: Build an async task queue

Create an async `TaskQueue` that processes tasks with configurable concurrency:

```python
queue = TaskQueue(max_concurrent=3)
queue.add(task_1)
queue.add(task_2)
results = await queue.run()
```

In [ ]:
# YOUR SOLUTION HERE


In [ ]:
# SOLUTION
import asyncio
from typing import Coroutine, Any

class TaskQueue:
    def __init__(self, max_concurrent: int = 5):
        self.max_concurrent = max_concurrent
        self._tasks: list[Coroutine] = []
    
    def add(self, coro_func, *args, **kwargs):
        self._tasks.append((coro_func, args, kwargs))
        return self
    
    async def run(self) -> list[Any]:
        sem = asyncio.Semaphore(self.max_concurrent)
        results = [None] * len(self._tasks)
        
        async def worker(index, coro_func, args, kwargs):
            async with sem:
                results[index] = await coro_func(*args, **kwargs)
        
        await asyncio.gather(
            *(worker(i, fn, args, kw) for i, (fn, args, kw) in enumerate(self._tasks))
        )
        return results

# Test it
async def process_item(item_id, delay=0.2):
    await asyncio.sleep(delay)
    return f"item_{item_id}_done"

queue = TaskQueue(max_concurrent=3)
for i in range(8):
    queue.add(process_item, i)

start = asyncio.get_event_loop().time()
results = await queue.run()
elapsed = asyncio.get_event_loop().time() - start

print(f"Results: {results}")
print(f"Completed in {elapsed:.2f}s (8 tasks, 3 concurrent, 0.2s each)")

### Problem 3: Implement `groupby` from scratch

Write a function that groups elements by a key function (like SQL GROUP BY).

```python
data = [{"dept": "eng", "name": "Alice"}, {"dept": "eng", "name": "Bob"}, {"dept": "sales", "name": "Charlie"}]
grouped = group_by(data, lambda x: x["dept"])
# {"eng": [{...Alice}, {...Bob}], "sales": [{...Charlie}]}
```

In [ ]:
# YOUR SOLUTION HERE


In [ ]:
# SOLUTION
from collections import defaultdict
from typing import TypeVar, Callable, Iterable

T = TypeVar('T')
K = TypeVar('K')

def group_by(iterable: Iterable[T], key_func: Callable[[T], K]) -> dict[K, list[T]]:
    groups: dict[K, list[T]] = defaultdict(list)
    for item in iterable:
        groups[key_func(item)].append(item)
    return dict(groups)

# Test with employees
employees = [
    {"name": "Alice", "dept": "Engineering", "salary": 120000},
    {"name": "Bob", "dept": "Engineering", "salary": 110000},
    {"name": "Charlie", "dept": "Sales", "salary": 90000},
    {"name": "Diana", "dept": "Sales", "salary": 95000},
    {"name": "Eve", "dept": "Engineering", "salary": 130000},
]

by_dept = group_by(employees, lambda e: e["dept"])
for dept, members in by_dept.items():
    names = [m["name"] for m in members]
    avg_salary = sum(m["salary"] for m in members) / len(members)
    print(f"{dept}: {names} (avg salary: ${avg_salary:,.0f})")

# Group numbers by even/odd
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
by_parity = group_by(numbers, lambda n: "even" if n % 2 == 0 else "odd")
print(f"\nBy parity: {by_parity}")

### Problem 4: Thread-safe counter with context manager

Create a thread-safe counter that uses a context manager to time operations and a lock to prevent race conditions.

```python
counter = ThreadSafeCounter()
# Run 1000 increments across 10 threads
# Final value should always be exactly 1000
```

In [ ]:
# YOUR SOLUTION HERE


In [ ]:
# SOLUTION
import threading
import time
from concurrent.futures import ThreadPoolExecutor

class ThreadSafeCounter:
    def __init__(self):
        self._value = 0
        self._lock = threading.Lock()
        self._start_time = None
    
    @property
    def value(self):
        return self._value
    
    def increment(self, amount=1):
        with self._lock:
            self._value += amount
    
    def decrement(self, amount=1):
        with self._lock:
            self._value -= amount
    
    def __enter__(self):
        self._start_time = time.perf_counter()
        return self
    
    def __exit__(self, *args):
        elapsed = time.perf_counter() - self._start_time
        print(f"Final value: {self._value}, Time: {elapsed:.4f}s")
        return False

with ThreadSafeCounter() as counter:
    def worker():
        for _ in range(100):
            counter.increment()
    
    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = [executor.submit(worker) for _ in range(10)]
        for f in futures:
            f.result()

print(f"Expected: 1000, Got: {counter.value}, Correct: {counter.value == 1000}")

### Problem 5: What's the output? (asyncio + generators)

Predict the output:

```python
import asyncio

async def task(name, delay):
    print(f"{name} start")
    await asyncio.sleep(delay)
    print(f"{name} end")
    return name

async def main():
    t1 = asyncio.create_task(task("A", 0.3))
    t2 = asyncio.create_task(task("B", 0.1))
    t3 = asyncio.create_task(task("C", 0.2))
    
    result = await t1
    print(f"Got: {result}")
    
    results = await asyncio.gather(t2, t3)
    print(f"Gathered: {results}")
```

In [ ]:
# Think about it, then run to verify

import asyncio

async def task(name, delay):
    print(f"{name} start")
    await asyncio.sleep(delay)
    print(f"{name} end")
    return name

async def main():
    t1 = asyncio.create_task(task("A", 0.3))
    t2 = asyncio.create_task(task("B", 0.1))
    t3 = asyncio.create_task(task("C", 0.2))
    
    result = await t1
    print(f"Got: {result}")
    
    results = await asyncio.gather(t2, t3)
    print(f"Gathered: {results}")

await main()

# Explanation:
# All 3 tasks start immediately when create_task is called.
# create_task schedules them — they begin at the next await point.
# Output:
#   A start
#   B start
#   C start
#   B end       (B finishes first at 0.1s)
#   C end       (C finishes next at 0.2s)
#   A end       (A finishes last at 0.3s)
#   Got: A      (we were awaiting t1)
#   Gathered: ['B', 'C']  (already completed, gather returns immediately)

### Problem 6: Implement a `pipe` function for data transformations

Create a `pipe` function that chains transformations, using `functools` and `itertools`:

```python
result = pipe(
    range(20),
    lambda data: filter(lambda x: x % 2 == 0, data),
    lambda data: map(lambda x: x ** 2, data),
    lambda data: itertools.takewhile(lambda x: x < 200, data),
    list
)
# [0, 4, 16, 36, 64, 100, 144, 196]
```

In [ ]:
# YOUR SOLUTION HERE


In [ ]:
# SOLUTION
from functools import reduce
import itertools

def pipe(data, *transforms):
    """Apply a series of transformations to data."""
    return reduce(lambda acc, fn: fn(acc), transforms, data)

result = pipe(
    range(20),
    lambda data: filter(lambda x: x % 2 == 0, data),
    lambda data: map(lambda x: x ** 2, data),
    lambda data: itertools.takewhile(lambda x: x < 200, data),
    list
)
print(f"Result: {result}")

# More readable with named functions
def only_even(data):
    return (x for x in data if x % 2 == 0)

def square(data):
    return (x ** 2 for x in data)

def below(threshold):
    return lambda data: itertools.takewhile(lambda x: x < threshold, data)

result = pipe(range(20), only_even, square, below(200), list)
print(f"Named: {result}")